In [11]:
# Importar las librerías necesarias

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [12]:
#### DEFINIMOS ACTIVOS Y PERÍODO ####

# En este caso, vamos a analizar dos bancos españoles cuando cotizan en el mercado continuo español
# (SAN y BBVA), el IBEX 35 gestionado por Bolsas y Mercados Españoles y el S&P 500 (Standard & Poor's 500,
# índice que opera principalmente en NYSE y NASDAQ) 
tickers = ["SAN.MC", "BBVA.MC", "^IBEX", "^GSPC"]
start = "2020-01-01"
end = "2025-01-01"

# Descargamos los precios de cierre
data = yf.download(tickers, start=start, end=end)["Close"]

print(f"✅ Datos descargados correctamente")
print(f"Periodo: {data.index[0].date()} → {data.index[-1].date()}")
print(f"Filas: {data.shape[0]} días de mercado")
data.tail()

[*********************100%***********************]  4 of 4 completed

✅ Datos descargados correctamente
Periodo: 2020-01-02 → 2024-12-31
Filas: 1292 días de mercado


Ticker,BBVA.MC,SAN.MC,^GSPC,^IBEX
Date,,,,
2024-12-24,8.751923,4.193863,6040.040039,11473.900391
2024-12-26,NaN,NaN,6037.589844,NaN
2024-12-27,8.865265,4.258891,5970.839844,11531.599609
2024-12-30,8.850153,4.267141,5906.939941,11536.799805
2024-12-31,8.927604,4.333140,5881.629883,11595.000000


In [ ]:
# Resumen estadístico de los precios
data.describe()

Ticker,BBVA.MC,SAN.MC,^GSPC,^IBEX
count,1281.000000,1281.000000,1258.000000,1281.000000
mean,5.004803,2.835115,4259.606933,8990.068220
std,2.243204,0.822146,767.448592,1318.580269
min,1.616620,1.233877,2237.399902,6107.200195
25%,3.444378,2.315560,3819.052551,8163.100098
50%,4.287387,2.727063,4204.880127,8871.099609
75%,6.532125,3.270211,4603.890137,9573.599609
max,10.063813,4.629099,6090.270020,12118.700195


In [ ]:
# Comprobamos si hay valores nulos
data.isnull().sum()

Ticker
BBVA.MC    11
SAN.MC     11
^GSPC      34
^IBEX      11
dtype: int64

In [ ]:
# Tipos de datos
data.dtypes

Ticker
BBVA.MC    float64
SAN.MC     float64
^GSPC      float64
^IBEX      float64
dtype: object

In [22]:
### CÁLCULO DE RETORNOS ACUMULADOS ###

# Hacemos aquí un tratamiento de los nulos para evitar que afecten a los cálculos posteriores. 
# Rellenamos hacia adelante los días sin cotización para cubrir diferencias entre los activos
data_filled = data.ffill()

# Retornos diarios
# El método pct_change() calcula la variación porcentual de cada día respecto al anterior.
# Luego, dropna() elimina la primera fila que queda con valor nulo al no tener un día anterior 
# para comparar.
returns = data_filled.pct_change().dropna()

# Retorno acumulado (base 100)
cumulative = (1 + returns).cumprod() * 100

print("\nRetorno total por activo (1 enero 2020- 1 enero 2025):")

# Se toma el valor de la última fila del DataFrame, que contiene el valor final en el que se ha 
# convertido la inversión inicial de 100. Luego, se resta 100 para obtener la ganancia o pérdida
# neta.
total_return = (cumulative.iloc[-1] - 100).round(2)
for activo, retorno in total_return.items():
    signo = "📈" if retorno > 0 else "📉"
    print(f"  {signo} {activo}: {retorno:+.1f}%")


Retorno total por activo (1 enero 2020- 1 enero 2025):
  📈 BBVA.MC: +146.7%
  📈 SAN.MC: +39.7%
  📈 ^GSPC: +80.5%
  📈 ^IBEX: +19.6%


Realizo una comprobación para comprobar que el retorno total coincide con el retorno acumulado:

In [14]:
print("Comprobación del retorno total por activo:\n")

for ticker in tickers:
    precio_inicial = data[ticker].dropna().iloc[0]
    precio_final = data[ticker].dropna().iloc[-1]
    retorno_manual = ((precio_final - precio_inicial) / precio_inicial * 100).round(2)
    print(f"{ticker}:")
    print(f"  Precio inicial: {precio_inicial:.2f}")
    print(f"  Precio final:   {precio_final:.2f}")
    print(f"  Retorno:        {retorno_manual:+.1f}%")
    print()

Comprobación del retorno total por activo:

SAN.MC:
  Precio inicial: 3.10
  Precio final:   4.33
  Retorno:        +39.7%

BBVA.MC:
  Precio inicial: 3.62
  Precio final:   8.93
  Retorno:        +146.7%

^IBEX:
  Precio inicial: 9691.20
  Precio final:   11595.00
  Retorno:        +19.6%

^GSPC:
  Precio inicial: 3257.85
  Precio final:   5881.63
  Retorno:        +80.5%



Ahora nos centramos en analizar la volatilidad de cada índice. 

In [47]:
### ANÁLISIS DE VOLATILIDAD ###

# Días reales de cotización por activo (excluimos los ceros del ffill)
dias_reales = (returns != 0).sum()/5  # Dividimos entre 5 para convertir a días/año aproximados

# Calculamos la volatilidad anualizada activo por activo
# volatilidad anualizada = desviación típica diaria * raíz cuadrada de los días reales de cotización * 100 (para convertir a porcentaje)
volatilidades = {}

for activo in returns.columns:
    # Paso 1: extraemos solo los retornos distintos de cero de este activo
    retornos_reales = returns[activo][returns[activo] != 0]
    
    # Paso 2: calculamos la desviación típica de esos retornos
    std_diaria = retornos_reales.std()

    # Paso 3: anualizamos usando los días reales de cotización de este activo
    factor_anualizacion = dias_reales[activo] ** 0.5
    
    # Paso 4: convertimos a porcentaje y guardamos
    volatilidades[activo] = round(std_diaria * factor_anualizacion * 100, 2)

# Convertimos el diccionario a Serie de pandas
volatility = pd.Series(volatilidades)

# Mostramos resultados
print("Volatilidad anualizada por activo:")
for activo, vol in volatility.items():
    dias_anuales = round(dias_reales[activo] / 5, 1)
    print(f"  {activo}: {vol}% (basado en {dias_anuales} días/año)")

# Volatilidad móvil a 30 días (rolling volatility)
rolling_vol = pd.DataFrame()

for activo in returns.columns:
    # Paso 1: copiamos los retornos y sustituimos ceros por None
    retornos_reales = returns[activo].copy()
    retornos_reales[retornos_reales == 0] = None
    
    # Paso 2: factor de anualización usando días ANUALES (igual que en volatility)
    factor = (dias_reales[activo] / 5) ** 0.5
    
    # Paso 3: calculamos la volatilidad móvil
    rolling_vol[activo] = retornos_reales.rolling(30).std() * factor * 100

print("✅ Volatilidad móvil calculada")


Volatilidad anualizada por activo:
  BBVA.MC: 38.66% (basado en 50.9 días/año)
  SAN.MC: 37.55% (basado en 51.1 días/año)
  ^GSPC: 21.31% (basado en 50.3 días/año)
  ^IBEX: 20.87% (basado en 51.1 días/año)
✅ Volatilidad móvil calculada


Realizamos unas comprobaciones sobre la validez de los cálculos anteriores.

In [48]:
print("Verificación de volatilidad móvil a 30 días - pico COVID (31 marzo 2020):\n")

for activo in returns.columns:
    # Paso 1: tomamos exactamente los últimos 30 retornos reales hasta el 31 de marzo 2020
    retornos_hasta_fecha = returns[activo][:"2020-03-31"]
    retornos_reales = retornos_hasta_fecha[retornos_hasta_fecha != 0]
    ultimos_30 = retornos_reales.tail(30)
    
    # Paso 2: calculamos la desviación típica de esos 30 retornos
    std_manual = ultimos_30.std()
    
    # Paso 3: anualizamos usando los días anuales medios
    factor = (dias_reales[activo] / 5) ** 0.5
    
    # Paso 4: convertimos a porcentaje
    vol_manual = round(std_manual * factor * 100, 2)
    
    # Paso 5: recuperamos el valor que calculó rolling
    vol_rolling = round(rolling_vol[activo]["2020-03-31"], 2)
    
    # Comparamos
    diferencia = abs(vol_manual - vol_rolling)
    coincide = "✅" if diferencia < 0.1 else "❌"
    
    print(f"{activo}: {coincide}")
    print(f"  Manual:     {vol_manual}%")
    print(f"  Rolling:    {vol_rolling}%")
    print(f"  Diferencia: {diferencia:.2f}%")
    print()

Verificación de volatilidad móvil a 30 días - pico COVID (31 marzo 2020):

BBVA.MC: ✅
  Manual:     40.02%
  Rolling:    40.02%
  Diferencia: 0.00%

SAN.MC: ✅
  Manual:     39.57%
  Rolling:    39.57%
  Diferencia: 0.00%

^GSPC: ✅
  Manual:     36.22%
  Rolling:    36.22%
  Diferencia: 0.00%

^IBEX: ✅
  Manual:     30.71%
  Rolling:    30.71%
  Diferencia: 0.00%



In [36]:
# Verificación: volatilidad solo en 2023 (año más tranquilo)
returns_2023 = returns["2023-01-01":"2023-12-31"]
dias_2023 = (returns_2023 != 0).sum()

print("Volatilidad anualizada en 2023 (año de referencia):")
for activo in returns.columns:
    retornos_reales = returns_2023[activo][returns_2023[activo] != 0]
    std_diaria = retornos_reales.std()
    factor = dias_2023[activo] ** 0.5
    vol_2023 = round(std_diaria * factor * 100, 2)
    print(f"  {activo}: {vol_2023}%")

Volatilidad anualizada en 2023 (año de referencia):
  BBVA.MC: 27.65%
  SAN.MC: 28.63%
  ^GSPC: 13.04%
  ^IBEX: 14.01%


In [40]:
# Verificación: volatilidad solo en 2020 (año más tranquilo)
returns_2020 = returns["2020-01-01":"2020-12-31"]
dias_2020 = (returns_2020 != 0).sum()

print("Volatilidad anualizada en 2020 (año de referencia):")
for activo in returns.columns:
    retornos_reales = returns_2020[activo][returns_2020[activo] != 0]
    std_diaria = retornos_reales.std()
    factor = dias_2020[activo] ** 0.5
    vol_2020 = round(std_diaria * factor * 100, 2)
    print(f"  {activo}: {vol_2020}%")

Volatilidad anualizada en 2020 (año de referencia):
  BBVA.MC: 60.11%
  SAN.MC: 58.12%
  ^GSPC: 34.49%
  ^IBEX: 34.01%


In [37]:
# Verificación: volatilidad solo en 2021 (año más tranquilo)
returns_2021 = returns["2021-01-01":"2021-12-31"]
dias_2021 = (returns_2021 != 0).sum()

print("Volatilidad anualizada en 2021 (año de referencia):")
for activo in returns.columns:
    retornos_reales = returns_2021[activo][returns_2021[activo] != 0]
    std_diaria = retornos_reales.std()
    factor = dias_2021[activo] ** 0.5
    vol_2021 = round(std_diaria * factor * 100, 2)
    print(f"  {activo}: {vol_2021}%")

Volatilidad anualizada en 2021 (año de referencia):
  BBVA.MC: 33.88%
  SAN.MC: 30.77%
  ^GSPC: 13.1%
  ^IBEX: 16.34%


In [38]:
# Verificación: volatilidad solo en 2024
returns_2024 = returns["2024-01-01":"2024-12-31"]
dias_2024 = (returns_2024 != 0).sum()

print("Volatilidad anualizada en 2024 (año de referencia):")
for activo in returns.columns:
    retornos_reales = returns_2024[activo][returns_2024[activo] != 0]
    std_diaria = retornos_reales.std()
    factor = dias_2024[activo] ** 0.5
    vol_2024 = round(std_diaria * factor * 100, 2)
    print(f"  {activo}: {vol_2024}%")

Volatilidad anualizada en 2024 (año de referencia):
  BBVA.MC: 28.24%
  SAN.MC: 25.71%
  ^GSPC: 12.65%
  ^IBEX: 13.34%


In [ ]:
# Verificación: volatilidad solo en 2022
returns_2022 = returns["2022-01-01":"2022-12-31"]
dias_2022 = (returns_2022 != 0).sum()

print("Volatilidad anualizada en 2022 (año de referencia):")
for activo in returns.columns:
    retornos_reales = returns_2022[activo][returns_2022[activo] != 0]
    std_diaria = retornos_reales.std()
    factor = dias_2022[activo] ** 0.5
    vol_2022 = round(std_diaria * factor * 100, 2)
    print(f"  {activo}: {vol_2022}%")

Volatilidad anualizada en 2022 (año de referencia):
  BBVA.MC: 34.2%
  SAN.MC: 35.49%
  ^GSPC: 24.13%
  ^IBEX: 19.61%


In [42]:
# Volatilidades año a año que ya calculamos
volatilidades_anuales = {
    "2020": {"BBVA.MC": 60.11, "SAN.MC": 58.12, "^GSPC": 34.49, "^IBEX": 34.01},
    "2021": {"BBVA.MC": 33.88, "SAN.MC": 30.77, "^GSPC": 13.10, "^IBEX": 16.34},
    "2022": {"BBVA.MC": 34.20, "SAN.MC": 35.49, "^GSPC": 24.13, "^IBEX": 19.61},
    "2023": {"BBVA.MC": 27.65, "SAN.MC": 28.63, "^GSPC": 13.04, "^IBEX": 14.01},
    "2024": {"BBVA.MC": 28.24, "SAN.MC": 25.71, "^GSPC": 12.65, "^IBEX": 13.34},
}

df_vol_anual = pd.DataFrame(volatilidades_anuales).T

print("Promedio ponderado de volatilidad (via varianza):\n")
for activo in df_vol_anual.columns:
    # Paso 1: convertimos volatilidades a varianzas (elevamos al cuadrado)
    varianzas = df_vol_anual[activo] ** 2
    
    # Paso 2: calculamos la media de las varianzas
    media_varianzas = varianzas.mean()
    
    # Paso 3: volvemos a volatilidad haciendo la raíz cuadrada
    vol_ponderada = round(media_varianzas ** 0.5, 2)
    
    # Comparamos con el resultado de la celda anterior
    vol_total = volatility[activo]
    
    coincide = "✅" if abs(vol_ponderada - vol_total) < 1 else "❌"
    print(f"{activo}: {coincide}")
    print(f"  Promedio via varianza:   {vol_ponderada}%")
    print(f"  Cálculo periodo total:   {vol_total}%")
    print()

Promedio ponderado de volatilidad (via varianza):

BBVA.MC: ✅
  Promedio via varianza:   38.71%
  Cálculo periodo total:   38.66%

SAN.MC: ✅
  Promedio via varianza:   37.59%
  Cálculo periodo total:   37.55%

^GSPC: ✅
  Promedio via varianza:   21.32%
  Cálculo periodo total:   21.31%

^IBEX: ✅
  Promedio via varianza:   20.89%
  Cálculo periodo total:   20.87%



In [8]:
### CORRELACIÓN ENTRE ACTIVOS ###

In [9]:
### RENTABILIDAD AJUSTADA AL RIESGO (SHARPE RATIO) ###

In [10]:
### VISUALIZACIÓN INTERACTIVA CON PLOTLY ###